## Imports

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_score, accuracy_score, f1_score, recall_score, top_k_accuracy_score


from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [ ]:
df = pd.read_csv('../src/data/pokedex.csv')
display(df.columns)
display(df)

## Feature engineering

In [ ]:
# Pour le modèle, on veut l'id du Pokémon et son/ses type(s) (variable cible)
# On récupère également le poids, la taille, l'expérience de base, le bonheur de base, les légendaires et les mythiques
# On conserve aussi capture_rate, gender_rate, growth_rate et hatch_counter (on les supprimera si nécessaire)
df_model = df[["id", "type_1", "type_2", "height", "weight", "base_experience", "base_happiness", "is_legendary", "is_mythical", "capture_rate", "gender_rate", "growth_rate", "hatch_counter"]].copy()

# On rajoute is_chimera et is_paradox
chimeras = [793, 794, 795, 796, 797, 798, 799, 803, 804, 805, 806]
paradoxs = [984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 1005, 1006, 1007, 1008, 1009, 1010, 1021, 1022, 1023]
df_model["is_chimera"] = df_model["id"].isin(chimeras)
df_model["is_paradox"] = df_model["id"].isin(paradoxs)


# On a besoin du total des statistiques de combat, ainsi que la répartition de ces statistiques
df_model["total_stats"] = df["hp"] + df["attack"] + df["defense"] + df["special-attack"] + df["special-defense"] + df["speed"]

stats = ["hp", "attack", "defense", "special-attack", "special-defense", "speed"]
for stat in stats:
	df_model[f"{stat}_pct"] = df[stat] * 100 / df_model["total_stats"]


# Il faut aussi la variation entre les différentes stats pour identifier les types les plus équilibrés
df_model["stats_variation"] = df[stats].std(axis=1) / df[stats].mean(axis=1) * 100


display(df_model.sort_values("id"))

In [ ]:
# Le total des statistiques peut aussi varier selon le niveau d'évolution du Pokémon
df_evolutions = df[["id", "name_en", "evolution_level", "evolves_from"]].copy()
babies = df.loc[df["is_baby"], "name_en"]

# Les Pokémons bébés ont un niveau d'évolution à -1
df_evolutions.loc[df_evolutions["name_en"].isin(babies), "evolution_level"] = -1

# Les Pokémons de base qui ont une pré-évolution bébé (comme Pikachu) ont un niveau d'évolution à 0
df_evolutions.loc[(df_evolutions["evolution_level"] == 1) & (df_evolutions["evolves_from"].isin(babies)), "evolution_level"] = 0
df_evolutions.loc[(df_evolutions["evolution_level"] == 0) & (df_evolutions["evolves_from"]), "evolves_from"] = np.nan

# Les Pokémons évoluant d'un Pokémon lui-même évolué (non bébé) ont un niveau d'évolution à 2
pokemons_lvl1 = df_evolutions.loc[df_evolutions["evolution_level"] == 1, "name_en"]
df_evolutions.loc[(df_evolutions["evolution_level"] == 1) & (df_evolutions["evolves_from"].isin(pokemons_lvl1)), "evolution_level"] = 2


df_model["evolution_level"] = df_evolutions["evolution_level"].copy()
df_model = df_model.sort_values("id")

display(df_model)

In [ ]:
display(df_model.columns)

## Modèle simple

### 1. Split test-train

In [ ]:
X = df_model.drop(columns=['type_1', 'type_2', 'id'])
y = df_model['type_1']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

### 2. Modèles de classification

__Modèles non pertinents :__

* __Régression logistique__ : fonctionne mieux sur des classifications binaires (ici il y a 18 types) avec une relation linéaire entre les variables et la cible (pas le cas ici)
* __Naive Bayes__ : part du principe que toutes les variables sont indépendantes, or dans notre cas c'est la combinaison de variables qui permet d'identifier un type
* __Arbre de décision__ : modèle très basique et sensible à l'overfitting, il vaut mieux partir sur des modèles qui utilisent plusieurs arbres de décision (random forest, xgboost)
* __K-Nearest Neighbors__ : performant sur des petits-moyens datasets avec de nombreuses classes, mais sensible aux outliers et nécessité de clusters nets pour chaque classe, or de nombreux types ont des frontières floues (insecte-fée similaires, eau qui n'a pas de caractéristique distinctive...)

__Modèles à tester :__

* __SVM__ : cherche la meilleure séparation entre les différents groupes, moins performant sur de nombreuses classes aux frontières parfois floues. À paramétrer avec précision.
* __Random Forest__ : plusieurs arbres de décision parallèles, on garde le meilleur. Capte les interactions complexes, la non-linéarité, le bruit, et hiérarchise les variables. À paramétrer avec précision pour éviter l'overfitting.
* __XGBoost__ : plusieurs arbres de décision à la suite, chacun apprend des erreurs du précédent. Plus rapide et performant que Gradient Boosting, à paraméter avec précision pour éviter l'overfitting.

##### Fonctions

In [ ]:
def model_scoring(y_pred_train, y_pred_test):
	accuracy_train = accuracy_score(y_train, y_pred_train)
	precision_train = precision_score(y_train, y_pred_train, average='weighted')
	recall_train = recall_score(y_train, y_pred_train, average='weighted')
	f1_train = f1_score(y_train, y_pred_train, average='weighted')

	accuracy_test = accuracy_score(y_test, y_pred_test)
	precision_test = precision_score(y_test, y_pred_test, average='weighted')
	recall_test = recall_score(y_test, y_pred_test, average='weighted')
	f1_test = f1_score(y_test, y_pred_test, average='weighted')

	def round_score(score):
		return round(score * 100, 2)

	print("SCORES TRAIN :")
	print("   • Accuracy:", round_score(accuracy_train), "%")
	print("   • Precision:", round_score(precision_train), "%")
	print("   • Recall:", round_score(recall_train), "%")
	print("   • F1-Score:", round_score(f1_train), "%")

	print("\nSCORES TEST :")
	print("   • Accuracy:", round_score(accuracy_test), "%")
	print("   • Precision:", round_score(precision_test), "%")
	print("   • Recall:", round_score(recall_test), "%")
	print("   • F1-Score:", round_score(f1_test), "%")

In [ ]:
numerical_features = X.select_dtypes(include=["int", "float"]).columns.to_list()
boolean_features = X.select_dtypes(include=["bool"]).columns.to_list()
categorical_features = X.select_dtypes(include=["object"]).columns.to_list()

preprocessor = ColumnTransformer(
	transformers=[
		("num", StandardScaler(), numerical_features),
		("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
		("bool", "passthrough", boolean_features)
	]
)


##### A. Support Vector Machines

In [ ]:
svm_smote = ImbPipeline([
	("preprocessor", preprocessor),
	("smote", SMOTE(random_state=42, sampling_strategy="all")),
	("classifier", SVC(random_state=42, kernel="rbf", C=10000, gamma=0.0001, probability=True))
])

svm_smote.fit(X_train, y_train)

y_pred_train = svm_smote.predict_proba(X_train)
y_pred_test = svm_smote.predict_proba(X_test)

top3_train = top_k_accuracy_score(y_train, y_pred_train, k=3)
top3_test = top_k_accuracy_score(y_test, y_pred_test, k=3)
print(f"Top-3 Accuracy (train): {top3_train:.2f}")
print(f"Top-3 Accuracy (test): {top3_test:.2f}\n")

y_pred_train = svm_smote.predict(X_train)
y_pred_test = svm_smote.predict(X_test)

model_scoring(y_pred_train, y_pred_test)

# print("\nMétriques détaillées :")
# print(classification_report(y_train, y_pred_train, zero_division=0))

##### B. Random Forest

In [ ]:
forest_smote = ImbPipeline([
	("preprocessor", preprocessor),
	("smote", SMOTE(random_state=42, sampling_strategy="all")),
	("classifier", RandomForestClassifier(
		random_state=42,
		max_depth=4,
		n_estimators=500,
		max_features="sqrt",
		max_leaf_nodes=8
	))
])

forest_smote.fit(X_train, y_train)
y_pred_train = forest_smote.predict_proba(X_train)
y_pred_test = forest_smote.predict_proba(X_test)

top3_train = top_k_accuracy_score(y_train, y_pred_train, k=3)
top3_test = top_k_accuracy_score(y_test, y_pred_test, k=3)
print(f"Top-3 Accuracy (train): {top3_train:.2f}")
print(f"Top-3 Accuracy (test): {top3_test:.2f}\n")

y_pred_train = forest_smote.predict(X_train)
y_pred_test = forest_smote.predict(X_test)

model_scoring(y_pred_train, y_pred_test)

# print("\nMétriques détaillées :")
# print(classification_report(y_train, y_pred_train, zero_division=0))

##### C. XG Boost

In [ ]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.fit_transform(y_test)

xgb_smote = ImbPipeline([
	("preprocessor", preprocessor),
	("smote", SMOTE(random_state=42, sampling_strategy="all")),
	("classifier", XGBClassifier(
		random_state=42,
		n_estimators=300,
		max_depth=4,
		learning_rate=0.001,
		subsample=0.6,
		colsample_bytree=0.6,
		reg_alpha=0.1,
        reg_lambda=1,
        n_jobs=-1,
        objective='multi:softmax',
        eval_metric='mlogloss'
	))
])

xgb_smote.fit(X_train, y_train)
y_pred_train = xgb_smote.predict_proba(X_train)
y_pred_test = xgb_smote.predict_proba(X_test)

top3_train = top_k_accuracy_score(y_train, y_pred_train, k=3)
top3_test = top_k_accuracy_score(y_test, y_pred_test, k=3)
print(f"Top-3 Accuracy (train): {top3_train:.2f}")
print(f"Top-3 Accuracy (test): {top3_test:.2f}\n")

y_pred_train = xgb_smote.predict(X_train)
y_pred_test = xgb_smote.predict(X_test)

model_scoring(y_pred_train, y_pred_test)

# print("\nMétriques détaillées :")
# print(classification_report(y_train, y_pred_train, zero_division=0))